# 01 — Data and ECFP4 baseline

**Goal:** load CO-ADD PA, look at the class balance, run the classical
ECFP4 + tree baseline (XGBoost here) under a leakage-free out-of-fold
protocol.

Read `docs/BACKGROUND.md` §3 (Gen 2) before this notebook — that is the
generation this baseline belongs to.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
from qsar_tutorial.data import load_coadd_pa, stratified_folds, Dataset
from qsar_tutorial.featurizer import featurize_ecfp4
from qsar_tutorial.model import cross_validated_oof

ds = load_coadd_pa('../data/processed/coadd_pa.csv')
print(f'N = {len(ds)} | active fraction = {ds.active_fraction:.1%}')

### Featurize with ECFP4 (the Gen-2 champion baseline)

In [ ]:
X, valid = featurize_ecfp4(list(ds.smiles), n_bits=2048)
X = X[valid]; y = ds.y[valid]
print(X.shape, 'active =', y.mean())

### Stratified 5-fold OOF

The metric you report is the **out-of-fold** AUC, not in-fold. This
is what makes the comparison honest.

In [ ]:
folds = stratified_folds(Dataset(smiles=ds.smiles[valid], y=y), n_splits=5)
oof = cross_validated_oof(X, y, folds, n_estimators=200, max_depth=6)
print(oof.summary())

### What to expect

Typical CO-ADD PA numbers for ECFP4+XGB:
- AUC ≈ 0.80–0.85
- AUPRC ≈ 0.30–0.45 (class imbalance bounds this)
- MCC@0.5 ≈ 0.30–0.40

Beating this with the Uni-Mol foundation model is **not guaranteed** —
see `docs/BACKGROUND.md` §5.2 (H1) for the honest framing.